# Payer Clustering Walkthrough

This notebook walks through the statistical methodology behind Upstream's payer behavioral classification system.

Upstream classifies payers into behavioral archetypes — **Aggressive Denier**, **Slow Payer**, **Prompt Payer**, **Underpayer** — by clustering their observed denial and payment behavior across claims. This is the statistical foundation of the 'behavioral fingerprinting' that powers the `check_payer_behavior` MCP tool.

**What you'll learn:**
- How to engineer payer-level behavioral features from 835 remittance data
- How K-means clustering separates payers into meaningful behavioral archetypes
- How to interpret and label the resulting clusters
- Why this is harder than it looks in production (and what the reference omits)

**Prerequisites:**
- CMS SynPUF sample data OR run `python sample_claims_schema.py` for synthetic data
- `pip install -r requirements.txt` from the repo root

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler

import sys
sys.path.insert(0, '..')

from reference.denial_clustering import cluster_payers, label_clusters, build_payer_features
from sample_claims_schema import generate_synthetic_claims

print('Dependencies loaded.')

## 1. Load Data

We use the synthetic claims generator here. To use real CMS SynPUF data, replace this cell with:
```python
df = pd.read_csv('../data/cms_synpuf_claims.csv')
```

The schema required: `payer_name`, `is_denied`, `denial_code`, `paid_amount`, `billed_amount`, `service_date`, `payment_date`.

In [ ]:
# Generate 2,000 synthetic claims across 8 fictional payers
df = generate_synthetic_claims(n_rows=2000, n_payers=8, seed=42)

print(f'Claims loaded: {len(df):,}')
print(f'Payers: {df.payer_name.nunique()}')
print(f'Overall denial rate: {df.is_denied.mean():.1%}')
df.head()

## 2. Engineer Payer-Level Behavioral Features

The key insight: individual claims are noisy. Behavioral patterns emerge at the payer level over time.

We aggregate each payer into a feature vector capturing:
- **denial_rate**: what fraction of claims get denied
- **avg_days_to_payment**: how long they take to pay (slow payer signal)
- **underpayment_rate**: claims paid below 95% of billed amount
- **appeal_success_rate**: what fraction of appeals reverse the denial
- **denial_rate_trend**: change in denial rate over the last 30 days (drift signal)

In [ ]:
payer_features = build_payer_features(df)
print(f'Payer profiles built: {len(payer_features)}')
payer_features

## 3. Scale and Cluster

K-means requires standardized features — denial rate (0–1) and days to payment (0–90) are on completely different scales. We use `StandardScaler` before clustering.

We use **K=4** here corresponding to the 4 production archetypes. In production, the optimal K is determined per-specialty using the elbow method on a rolling 90-day window.

In [ ]:
clustered = cluster_payers(df, n_clusters=4)
labeled = label_clusters(clustered)

print('Cluster assignments:')
print(labeled[['payer_name', 'cluster', 'cluster_label', 'denial_rate', 'avg_days_to_payment']].to_string())

## 4. Visualize the Clusters

Plot payers in denial-rate × payment-speed space. The four quadrants map to the behavioral archetypes:

- High denial + slow payment → **Aggressive Denier**
- Low denial + slow payment → **Slow Payer**  
- Low denial + fast payment → **Prompt Payer**
- Low denial + fast payment + high underpayment → **Underpayer** (not visible in 2D)

In [ ]:
COLOR_MAP = {
    'Aggressive Denier': '#EF4444',
    'Slow Payer': '#F5A623',
    'Prompt Payer': '#22C55E',
    'Underpayer': '#A78BFA',
}

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('#0A1628')
ax.set_facecolor('#0D1B2A')

for _, row in labeled.iterrows():
    color = COLOR_MAP.get(row['cluster_label'], '#888888')
    ax.scatter(
        row['denial_rate'],
        row['avg_days_to_payment'],
        c=color,
        s=200,
        zorder=5,
        edgecolors='white',
        linewidth=0.5,
    )
    ax.annotate(
        row['payer_name'],
        (row['denial_rate'], row['avg_days_to_payment']),
        textcoords='offset points',
        xytext=(8, 4),
        color='#FFFBF5',
        fontsize=9,
    )

# Add quadrant lines
ax.axvline(x=labeled['denial_rate'].median(), color='#FFFFFF22', linestyle='--', linewidth=1)
ax.axhline(y=labeled['avg_days_to_payment'].median(), color='#FFFFFF22', linestyle='--', linewidth=1)

ax.set_xlabel('Denial Rate', color='#FFFBF5', fontsize=11)
ax.set_ylabel('Avg Days to Payment', color='#FFFBF5', fontsize=11)
ax.set_title('Payer Behavioral Clusters', color='#FFFBF5', fontsize=14, pad=14)
ax.tick_params(colors='#FFFBF5')
for spine in ax.spines.values():
    spine.set_edgecolor('#FFFFFF33')

patches = [mpatches.Patch(color=c, label=l) for l, c in COLOR_MAP.items()]
ax.legend(handles=patches, facecolor='#0D1B2A', edgecolor='#FFFFFF33', labelcolor='#FFFBF5')

plt.tight_layout()
plt.show()

## 5. What Production Does Differently

This reference implementation illustrates the core idea. Production adds:

1. **Temporal clustering**: The model runs on rolling 30/60/90-day windows. A payer that was Prompt Payer in Q1 and is now Aggressive Denier shows a behavioral shift — that shift is the signal.

2. **Cross-customer pooling**: Upstream clusters across 200+ practices simultaneously. A payer that looks like Aggressive Denier to 31 different practices is a network-wide event, not a billing problem.

3. **Specialty-specific thresholds**: A 23% denial rate means something very different for ABA (where 20–30% is normal) vs SNF (where >5% is alarming). Production uses specialty-stratified clustering.

4. **Feature depth**: Production uses 40+ features including auth policy change flags, adjudication criterion deltas, and RAC audit signals. This reference uses 5.

The statistical test for 'did this payer just change their behavior?' is in `drift_detection_reference.py`.

In [ ]:
# Summary table
summary = labeled.groupby('cluster_label').agg(
    n_payers=('payer_name', 'count'),
    avg_denial_rate=('denial_rate', 'mean'),
    avg_days_payment=('avg_days_to_payment', 'mean'),
).round(3)

print('Cluster Summary:')
print(summary.to_string())